In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder
from structural_analysis.Material import Material

# Loading the pre-computed planforms and the fuselage

In [2]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

assumptions = Assumptions()

In [ ]:
# --- Import Onshape pull utilities ---
sys.path.append(os.path.abspath(os.getcwd()))
from onshape_pull import (
    fetch_variable_studio, fetch_measurement_features,
    evaluate_measurements, load_cached_masses, fetch_mass_properties,
    compute_cg_scenarios, lookup_var, lookup_meas,
    UPDATE_MASSES,
)

# --- Pull data from Onshape ---
variables = fetch_variable_studio()
meas_names = fetch_measurement_features()
measurements = evaluate_measurements(meas_names)
components = load_cached_masses() if not UPDATE_MASSES else fetch_mass_properties()
cg_data = compute_cg_scenarios(components)

# --- Z offset (axle datum) ---
Front_Landing_Gear_Hinge_Z = lookup_meas(measurements, "Front_Landing_Gear_Hinge_Z")
Front_Strut_Height, _, _ = lookup_var(variables, "Front_Strut_Height")
Front_Gear_Extension_Max = lookup_meas(measurements, "Front_Gear_Extension_Max")
Front_Gear_Extension_Min = lookup_meas(measurements, "Front_Gear_Extension_Min")
z_offset = (Front_Landing_Gear_Hinge_Z + Front_Strut_Height
            + (Front_Gear_Extension_Max - Front_Gear_Extension_Min))

# --- Build drag components ---
engine_bay = Bay(
    surface_wetted=83744.32631 / 1e6,  # mm² → m² (hardcoded, not in Onshape)
    length=0.172,                       # 172 mm (hardcoded, not in Onshape)
    diameter=lookup_var(variables, "engine_diameter")[0],
)

Front_Gear_Unexposed = lookup_meas(measurements, "Front_Gear_Unexposed")
nose_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Front_Strut_Height - Front_Gear_Unexposed,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Front_Strut_Diameter")[0],
)

Rear_Strut_Height, _, _ = lookup_var(variables, "Rear_Strut_Height")
Rear_Strut_height_2, _, _ = lookup_var(variables, "Rear_Strut_height_2")
main_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Rear_Strut_Height + Rear_Strut_height_2,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Rear_Strut_Diameter")[0],
)

fuselage = Fuselage(
    surface_wetted=lookup_meas(measurements, "Wetted_Area"),
    length_total=lookup_var(variables, "FuselageLength")[0],
    diameter_max=lookup_var(variables, "FuselageHeight")[0],
    upsweep=0.0,
    base_area=lookup_meas(measurements, "Base_Area"),
)

# --- X-position helpers ---
WingPortDistance, _, _ = lookup_var(variables, "WingPortDistance")
WingPortWidth, _, _ = lookup_var(variables, "WingPortWidth")
CanardPortXLoc, _, _ = lookup_var(variables, "CanardPortXLoc")
CanardPortWidth, _, _ = lookup_var(variables, "CanardPortWidth")

# --- Fixed parameters ---
fixed = Fixed(
    mass=cg_data["mass"],
    fuel_mass=cg_data["fuel_mass"],
    x_cg_min=cg_data["x_cg_min"],
    x_cg_max=cg_data["x_cg_max"],
    x_tail_cone=lookup_meas(measurements, "Tailcone_X"),
    z_cg=cg_data["z_cg_full"] + z_offset,
    z_tail_cone=-lookup_meas(measurements, "Z_TailCone") + z_offset,
    z_wing=lookup_meas(measurements, "Z_wing_LE_Abs") + z_offset,
    x_LE_canard=CanardPortXLoc + CanardPortWidth / 2,
    x_LE_wing=WingPortDistance + 0.115 - WingPortWidth / 2,
    x_LE_tail=lookup_meas(measurements, "X_LE_Tail"),
    x_nose_gear=lookup_meas(measurements, "x_nose_gear"),
    x_main_gear=lookup_meas(measurements, "x_main_gear"),
    y_main_gear=0.419,
    fuselage=fuselage,
    nose_gear=nose_gear,
    main_gear=main_gear,
    engine_bay=engine_bay,
)

  → Found 78 occurrences, fetching mass properties...
    ... processed 10/78 occurrences
    ... processed 20/78 occurrences
    ... processed 30/78 occurrences
    ... processed 40/78 occurrences
    ... processed 50/78 occurrences
    ... processed 60/78 occurrences
    ... processed 70/78 occurrences
  → Mass data cached to mass_cache.json


In [4]:
with open("pickles/fixed_pickle.pcl", "wb") as f:
    pickle.dump(fixed, f)

In [5]:
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

In [6]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Creating full Aircraft objects

In [7]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

In [8]:
aircraft:list[Aircraft] = list()

for i, planform_recovered in enumerate(plaforms_recovered):
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=max(4., main_wing.aspect_ratio/2)) if (planform_type == "tail") else CanardFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_c=max(5., main_wing.aspect_ratio/2))
    
    emp = ef.find_planforms(main_wing, print_=i==28)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

0.2944239494738869
Stresses 509648.13360608864, 1564471.2029410438, 0.0004
0.294423949473887
Stresses 1543256.6835347144, 16087455.390532745, 0.0004
3.0483310160132437
Stresses 2940237.315930425, 8753540.2188407, 0.0004
3.048331016013243
Stresses 2736763.9529508976, 7572432.403767246, 0.0004
2.730777241832761
Stresses 2707419.2233016603, 8059570.10400153, 0.0004
2.7307772418327616
Stresses 2664787.412071453, 7804876.018769371, 0.0004
2.7502146612554186
Stresses 2721857.720147237, 8088734.803916801, 0.0004
2.7502146612554186
Stresses 2669373.997089161, 7776308.307393726, 0.0004
2.7489802245992907
Stresses 2720941.5169294756, 8106238.880325705, 0.0004
2.748980224599291
Stresses 2669083.456905402, 7796727.206856553, 0.0004
2.749019330479799
Stresses 2720970.543029023, 8088096.175910277, 0.0004
2.7490193304797987
Stresses 2669092.6625114493, 7779164.180897028, 0.0004
0.3415475767479746
Stresses 661665.8918851808, 2513038.8733308823, 0.0004
0.3412436975038259
Stresses 332337.3315191664, 983

In [9]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))
ac_bad:Aircraft = aircraft[np.argmax(s_ratios)],
print(np.argmax(s_ratios))
ac_bad= ac_bad[0]
print(ac_bad.planforms[0].sweep_quarter_rad, ac_bad.planforms[0].aspect_ratio, ac_bad.planforms[0].cm_quarter_chord, ac_bad.planforms[0].thickness_to_chord)

[0.30574772959985763, 0.2906157171037351, 0.1236569668390265, 0.11202914640080329, 0.7536014825453118, 0.5992833652656393, 0.5626093732299825, 0.41730985303965834, 0.3050854834942149, 0.2899461101945982, 0.12295037838587883, 0.11136120343305814, 0.7527903843554522, 0.5984600162588336, 0.5617386814878569, 0.4165240996055186, 0.304394597845672, 0.28924769354474694, 0.12221314468000319, 0.11066452402306107, 0.7519440843545142, 0.5976014088630293, 0.5608302156190366, 0.4157047161273902, 0.1367521825937778, 0.15447838203552072, 0.018109182631681417, 0.018953042859176573, 0.7186613397138906, 0.5761872168397364, 0.5898190725040868, 0.4508832577858616, 0.1418382884078466, 0.16047088652332073, 0.02355244842195363, 0.02492510954697773, 0.7257770298366478, 0.5834448593059532, 0.5972913586111579, 0.45791909729501845, 0.14403251403560346, 0.1630600162127525, 0.025894964815349097, 0.027505789643079745, 0.7288445258797488, 0.5865841793326927, 0.6005127828385892, 0.4609627540330851, 0.1304192125329183

# Checking if requirements are met

In [10]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [11]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()

Fuel available: 11.000000001437998 kg
Fuel required: 7.009019028012672 kg
Difference: 3.9909809734253265 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.022578454835737 kg
Difference: 3.977421546602261 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.172962019260426 kg
Difference: 3.827037982177572 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.110699205206417 kg
Difference: 3.8893007962315815 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 6.546244112929252 kg
Difference: 4.453755888508746 kg
all constraints satisfied
ac mass: 36.17975845823961, 0.7845238526961408
MainWing: AR=5.0, tc=0.06, sweep=14.999999999999998 deg, cmac=-0.15
Failed: ['Empennage Requirement']

Fuel available: 11.000000001437998 kg
Fuel required: 6.897015420418916 kg
Difference: 4.102984581019082 kg
all constraints satisfied
ac mass: 36.17975845823961, 0.7845238526961408
Mai